In [ ]:
# Importar librerías estándar para análisis de datos y visualización
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Herramientas de Scikit-learn: división de datos, escalado y métricas de regresión
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score

In [ ]:
# TensorFlow / Keras para construir y entrenar la red neuronal
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping

### Análisis exploratorio

In [ ]:
# Ruta al dataset de precios de viviendas (King County, USA)
file_path = '~/tensorflow_ejemplos/DATA/kc_house_data.csv'

In [ ]:
# Cargar el CSV en un DataFrame de pandas y mostrar las primeras filas
df = pd.read_csv(file_path)
df.head()

In [ ]:
# Verificar valores nulos por columna
df.isnull().sum()

In [ ]:
# Estadísticas descriptivas de todas las variables numéricas
df.describe().transpose()

In [ ]:
# Distribución de la variable objetivo (precio)
sns.histplot(data=df, x='price');

In [ ]:
# Conteo de propiedades según número de habitaciones
sns.countplot(data=df, x='bedrooms');

In [ ]:
# Relación entre precio y metros cuadrados de área habitable
sns.scatterplot(data=df, x='price', y='sqft_living');

In [ ]:
# Boxplot de precio vs número de habitaciones para detectar outliers
sns.boxplot(data=df, x='bedrooms', y='price');

In [ ]:
# Mapa geográfico de precios excluyendo el 1% de casas más caras (mejor visualización)
top_1 = int(len(df) * 0.01)
df_non_top_1 = df.sort_values('price', ascending=False).iloc[top_1:]
sns.scatterplot(data=df_non_top_1, x='long', y='lat', hue='price', palette='RdYlGn');

In [ ]:
# Comparación de precios entre propiedades con y sin vista al agua
sns.boxplot(data=df, x='waterfront', y='price');

In [ ]:
# Resumen de tipos de datos y uso de memoria
df.info()

### Ingeniería de Datos

In [ ]:
# Eliminar la columna 'id' porque no aporta información predictiva
df = df.drop('id', axis=1)
df.head()

In [ ]:
# Convertir la columna de fecha a tipo datetime y extraer mes/año como features
df['date'] = pd.to_datetime(df['date'])
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year

In [ ]:
# Boxplot de precio vs año de venta
sns.boxplot(data=df, x='year', y='price');

In [ ]:
# Evolución del precio promedio por mes (forma moderna de agrupar en pandas)
df.groupby('month')['price'].mean().plot();

In [ ]:
# Evolución del precio promedio por año
df.groupby('year')['price'].mean().plot();

In [ ]:
# Ya extraímos mes y año; descartamos la columna original de fecha
df = df.drop('date', axis=1)

In [ ]:
# Listar columnas actuales después de la limpieza
df.columns

In [ ]:
# Descartar 'zipcode': es categórica con alta cardinalidad y no se one-hot-encodea aquí
df = df.drop('zipcode', axis=1)

In [ ]:
# Verificar columnas finales antes de modelar
df.columns

### Escalado y Train-Test Split

In [ ]:
# Separar features (X) y variable objetivo (y)
X = df.drop('price', axis=1).values
y = df['price'].values

In [ ]:
# Dividir en entrenamiento (80%) y prueba (20%) con semilla para reproducibilidad
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=101)

In [ ]:
# Escalar features al rango [0, 1] usando solo el conjunto de entrenamiento
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)   # Ajustar y transformar train
X_test = scaler.transform(X_test)         # Transformar test con los parámetros de train

In [ ]:
# Forma del conjunto de entrenamiento: (muestras, features)
X_train.shape

In [ ]:
# Forma del conjunto de prueba
X_test.shape

### Crear Modelo de Entrenamiento

In [ ]:
# Definir arquitectura de la red neuronal usando la API Sequential moderna de Keras
# Se obtiene el número de features dinámicamente para no hardcodear dimensiones
input_dim = X_train.shape[1]

model = Sequential([
    Input(shape=(input_dim,)),               # Capa de entrada: shape = (features,)
    Dense(units=128, activation='relu'),     # Capa oculta 1: 128 neuronas + ReLU
    Dense(units=64, activation='relu'),      # Capa oculta 2: 64 neuronas + ReLU
    Dense(units=32, activation='relu'),      # Capa oculta 3: 32 neuronas + ReLU
    Dense(units=1)                           # Capa de salida: 1 neurona (regresión lineal)
])

### Elegir optimizador y función de pérdida

#### Para clasificación multiclase
`model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])`

#### Para clasificación binaria
`model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])`

#### Para regresión (MSE)
`model.compile(optimizer='adam', loss='mse')`

| Tipo                          | Función de pérdida principal                  | Optimizadores comunes       |
|-------------------------------|-----------------------------------------------|-----------------------------|
| **Regresión**                 | Mean Squared Error (MSE)                      | Adam, SGD, RMSprop          |
| **Clasificación binaria**     | Binary Cross-Entropy (BCE)                    | Adam, SGD con momentum      |
| **Clasificación multiclase**  | Categorical Cross-Entropy (CCE)               | Adam, RMSprop, Adagrad      |

In [ ]:
# Compilar el modelo: Adam es el optimizador por defecto en la mayoría de casos; MSE para regresión
model.compile(optimizer='adam', loss='mse')

In [ ]:
# Entrenar el modelo; usamos validation_data para monitorear overfitting en cada época
history = model.fit(
    x=X_train,
    y=y_train,
    validation_data=(X_test, y_test),
    batch_size=128,
    epochs=100
)

### Evaluación

In [ ]:
# Convertir el historial de entrenamiento a DataFrame y graficar la evolución de la pérdida
history_df = pd.DataFrame(history.history)
history_df.plot();

In [ ]:
# Generar predicciones sobre el conjunto de prueba
predictions = model.predict(X_test)

In [ ]:
# Error absoluto medio (MAE): interpretable en las mismas unidades del precio
mean_absolute_error(y_test, predictions)

In [ ]:
# Raíz del error cuadrático medio (RMSE): penaliza más los errores grandes
rmse = np.sqrt(mean_squared_error(y_test, predictions))
rmse

In [ ]:
# Explained Variance Score: 1.0 indica predicción perfecta; valores negativos indican muy mal desempeño
explained_variance_score(y_test, predictions)

In [ ]:
# Gráfico de predicciones vs valores reales; la línea roja representa la predicción perfecta
plt.scatter(y_test, predictions)
plt.plot(y_test, y_test, 'r')
plt.xlabel('Valor Real')
plt.ylabel('Predicción')
plt.title('Predicciones vs Valores Reales');

In [ ]:
# Dimensiones del vector de prueba
y_test.shape

In [ ]:
# Distribución de los errores de predicción (residuos)
errors = y_test.reshape(-1, 1) - predictions
sns.histplot(errors)
plt.xlabel('Error (Real - Predicción)')
plt.title('Distribución de Errores');

### Predecir un Nuevo Ítem

In [ ]:
# Seleccionar una casa aleatoria del DataFrame original para validar la predicción
np.random.seed(42)
house_num = np.random.randint(len(df))
single_house = df.drop('price', axis=1).iloc[house_num]

In [ ]:
# Escalar la muestra individual con el mismo scaler usado en entrenamiento
single_house = scaler.transform(single_house.values.reshape(1, -1))
single_house

In [ ]:
# Predecir el precio de la casa seleccionada
prediction = model.predict(single_house)
prediction

In [ ]:
# Mostrar los datos reales de la casa para comparar con la predicción
df.iloc[house_num]